In [1]:
import math
import numpy as np
from scipy.stats import chi2

def est_T(frecuencias: list[int], probabilidades: list[float], n: int, K: int):
    t = 0
    for k in range(K):
        t = t + (frecuencias[k] - n * probabilidades[k]
                 )**2 / (n * probabilidades[k])

    return t


def gen_frecuencias_binomial(probabilidades: list[float], n: int, K: int) -> list[int]:
    # Genera directamente las frecuencias para las variables discretas ya que
    # al no estimar ningún parámetro, no es necesario generar muestras

    n_res = n
    freq = []
    p = probabilidades[0]
    for i in range(K - 1):
        N_i = np.random.binomial(n_res, p)
        freq.append(N_i)

        n_res -= N_i
        p = probabilidades[i + 1] / (1 - sum(probabilidades[:(i + 1)]))

    freq.append(n_res)

    return freq

### Ejercicio 6.
Un escribano debe validar un juego en cierto programa de televisión. El mismo consiste en hacer girar una rueda y obtener un premio según el sector de la rueda que coincida con una aguja. Hay $10$ premios posibles, y las áreas de la rueda para los distintos premios, numerados del $1$ al $10$, son respectivamente:
$$ 31\%, \ 22\%, \ 12\%, \ 10\%, \ 8\%, \ 6\%, \ 4\%, \ 4\%, \ 2\%, \ 1\% $$

Los premios con número alto (e.j. un auto 0Km) son mejores que los premios con número bajo (e.j. 2x1 para entradas en el cine). El escribano hace girar la rueda hasta que se cansa, y anota cuántas veces sale cada sector. Los resultados, para los premios del $1$ al $10$, respectivamente, son:
$$ 188, 138, 87, 65, 48, 32, 30, 34, 13, 2 $$

a) Construya una tabla con los datos disponibles

b) Diseñe una prueba de hipótesis para determinar si la rueda es justa

c) Defina el *p-valor* a partir de la hipótesis nula

d) Calcule el *p-valor* bajo la hipótesis de que la rueda es justa, usando la aproximación chi cuadrado

e) Calcule el *p-valor* bajo la hipótesis de que la rueda es justa, usando una simulación.

---

### Análisis
Tenemos que
$$
\begin{array}{cl}

H_0 & = \text{"La rueda es justa"} = \text{"La rueda tiene las probabilidades conforme a su area"} \\
H_1 & = \text{"La rueda no es justa"} = \text{"La rueda no tiene las probabilidades conforme a su area"}

\end{array}
$$

Usamos la prueba chi-cuadrada Pearson con el estadístico:
$$
T = \sum_{i=1}^{K}\frac{(N_i - n p_i)^2}{n p_i}
$$

Donde K es la cantidad de valores que puede tener nuestra muestra discreta, se tiene que para cada $i\in \{1,2,\cdots,K\}$:

*   $N_i$ es la frecuencia observada en los datos correspondientes a la clase $i$.
*   $p_i$ es la probabilidad de que la variable aleatoria tome el valor $i$

El *p-valor* ($p$) viene dado por:
$$
p = P(T \geq t_{obs}) \sim P(χ_{k-1} \geq t_{obs})
$$

Asumo un $\alpha = 0.05$ para rechazar o no la $H_0$.

Si $t_{obs}$ es el estadístico obtenido con la muestra original, el *p-valor* se aproxima mediante simulaciones de la siguiente forma:

$$
p\text{-valor} \approx \frac{\#\{j:t_{(j)}\ge t_{obs}\}}{N_{sim}}.
$$

Siendo $t_{(j)}$ el valor al evaluar T con una muestra generada según F de $H_0$.

---

Nuestro $K$ es igual a $10$ (valores entre el 1 y el 10) porque la variable $X$ puede tomar $10$ valores distintos.

Ademas:
$$ \text{p-valor} \sim P(χ_{9} \geq t_{obs}) $$

In [2]:
frecuencias = [188, 138, 87, 65, 48, 32, 30, 34, 13, 2]
probabilidades = [0.31, 0.22, 0.12, 0.1, 0.08, 0.06, 0.04, 0.04, 0.02, 0.01]

K = len(frecuencias)
n = sum(frecuencias)

t_obs = est_T(frecuencias, probabilidades, n, K)
p_valor = chi2.sf(t_obs, df=(K - 1))

print("t_obs: ", t_obs)
print("p_valor: ", p_valor)

t_obs:  9.810370888711903
p_valor:  0.3660538998868262


Como $\text{p-valor} \gt \alpha \equiv 0.366 \gt 0.05$, entonces no hay suficiente evidencia para rechazar $H_0$.

In [3]:
N_SIM = 10_000

contador = 0
for _ in range(N_SIM):
    frec = gen_frecuencias_binomial(probabilidades, n, K)

    t_j = est_T(frec, probabilidades, n, K)
    if t_j >= t_obs:
        contador += 1

p_valor = contador / N_SIM
print("p_valor simulado: ", p_valor)

p_valor simulado:  0.3598


Como $\text{p-valor} \gt \alpha \equiv 0.3598 \gt 0.05$, entonces no hay suficiente evidencia para rechazar $H_0$.